# 02 — Condition-stratified differential expression

**Goal:** a log2FC signature for every (perturbation, condition) pair.

## The design constraint

Every contrast is perturbation vs. **condition-matched control**. Never pool
controls across conditions. IFN-γ stimulation and TIL co-culture shift the
baseline transcriptome enormously, and pooling would let that shift leak into
every perturbation's signature — producing a screen in which everything is a
hit and none of it means anything.

## The replication caveat

There are no biological replicates here. Pseudo-replicates are random splits
of one sample: they give the negative-binomial model a within-group variance
term, but systematically understate biological variance, so p-values are
anti-conservative. Treat the DE ranking as a *ranking*. The permutation-tested
E-distance in nb03 is the honest effect-size check.

In [ ]:
# =============================================================================
# nb02 — Condition-stratified differential expression
#
# Produces the core object of the project: a log2FC signature for every
# (perturbation, condition) pair, each computed against CONDITION-MATCHED
# control-guide cells.
#
# Replicate structure: sgRNAs. Each gene carries ~4 independent guides, and
# those are genuinely independent perturbation events — different cut sites,
# different off-target profiles, independently infected cells. Splitting cells
# randomly would give DESeq2 a variance term that reflects only sampling noise;
# splitting by guide gives it something closer to real experimental variance.
# =============================================================================

%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import load_config, load_panels, paths, set_seed
from src.plotting import apply_style, condition_palette, savefig

cfg    = load_config()
panels = load_panels()
P      = paths(cfg)
SEED   = set_seed(cfg)
apply_style(cfg)

sc.settings.verbosity = 1

# ---- schema shortcuts, so column names appear once ------------------------
s          = cfg["schema"]["obs"]
PERT       = s["perturbation"]        # "perturbation"    — CRISPR target
COND       = s["condition"]           # "perturbation_2"  — Control / IFNγ / Co-culture
GUIDE      = s["guide"]               # "sgRNA"           — replicate unit
CTRL       = cfg["schema"]["control_label"]
REF        = cfg["schema"]["conditions"]["reference"]
cond_order = ["Control", "IFNγ", "Co-culture"]
pal        = condition_palette(cfg)

# ---- load the paired object written by nb01 -------------------------------
import mudata as md
mdata = md.read(P.data_interim / "frangieh_qc.h5mu")
rna, adt = mdata["rna"], mdata["adt"]

print(f"repo:  {P.root}")
print(f"seed:  {SEED}")
print(f"RNA:   {rna.n_obs:,} cells x {rna.n_vars:,} genes")
print(f"ADT:   {adt.n_obs:,} cells x {adt.n_vars:,} features")
print(f"\ncells per condition:")
print(rna.obs[COND].value_counts().reindex(cond_order).to_string())
print(f"\nperturbations: {rna.obs[PERT].nunique()}  (incl. '{CTRL}')")
print(f"guides:        {rna.obs[GUIDE].nunique()}")

# ---- confirm we are on the filtered object --------------------------------
assert set(rna.obs["MOI"].unique()) == {1}, "expected MOI==1 cells only — re-run nb01 section 6"

In [ ]:
# =============================================================================
# Self-knockdown QC
#
# Cas9 frameshifts trigger nonsense-mediated decay, so a perturbation's own
# transcript should fall in its own arm. This is the CRISPR-KO analogue of the
# target-upregulation check used for CRISPRa, with the sign reversed.
#
# Editing happened BEFORE the condition treatments, so knockdown should look
# the same in all three arms. Divergence between conditions here would indicate
# a problem with the contrast logic, not biology.
#
# Caveat: absence of knockdown is ambiguous under Cas9. In-frame indels
# preserve the transcript, unedited alleles contribute normal message, and some
# genes escape NMD entirely (last-exon variants, short transcripts). A gene
# with no transcript loss may still be protein-null.
#
# Computed directly by pseudobulk rather than from the DE matrix — this is a
# single-gene question per perturbation, and it gates whether the ~750-contrast
# DESeq2 run is worth launching.
# =============================================================================

# ---- CPM-normalised pseudobulk per (perturbation, condition) ---------------
# Sum raw counts within each group, then normalise to counts-per-million so
# groups of different size are comparable. log2((cpm_pert + 1) / (cpm_ctrl + 1))
# against the CONDITION-MATCHED control — never pooled across conditions.
import scipy.sparse as sp

counts = rna.layers["counts"] if "counts" in rna.layers else rna.X
gene_ix = {g: i for i, g in enumerate(rna.var_names)}

groups = (rna.obs[PERT].astype(str) + "|" + rna.obs[COND].astype(str)).values
uniq = pd.unique(groups)
row_of = {g: i for i, g in enumerate(uniq)}

# one-hot (groups x cells) @ (cells x genes) -> summed counts per group
M = sp.csr_matrix(
    (np.ones(len(groups)), ([row_of[g] for g in groups], np.arange(len(groups)))),
    shape=(len(uniq), rna.n_obs),
)
pb = M @ counts
pb = np.asarray(pb.todense()) if sp.issparse(pb) else np.asarray(pb)
cpm = pb / pb.sum(axis=1, keepdims=True) * 1e6
cpm = pd.DataFrame(cpm, index=uniq, columns=rna.var_names)

# ---- self log2FC for every (target, condition) ----------------------------
rows = []
for cond in cond_order:
    ctrl_key = f"{CTRL}|{cond}"
    if ctrl_key not in cpm.index:
        print(f"[warn] no control group for {cond}")
        continue
    for pert in rna.obs[PERT].unique():
        if pert == CTRL:
            continue
        key = f"{pert}|{cond}"
        if key not in cpm.index or pert not in gene_ix:
            continue          # target not measured in this matrix
        rows.append({
            "perturbation": pert,
            "condition": cond,
            "cpm_pert": cpm.loc[key, pert],
            "cpm_ctrl": cpm.loc[ctrl_key, pert],
            "n_cells": int((groups == key).sum()),
        })

self_kd = pd.DataFrame(rows)
self_kd["self_log2fc"] = np.log2(
    (self_kd["cpm_pert"] + 1) / (self_kd["cpm_ctrl"] + 1)
)
print(f"{self_kd['perturbation'].nunique()} targets measurable in the RNA matrix "
      f"of {rna.obs[PERT].nunique() - 1} perturbed")

In [ ]:
# =============================== FIGURE ====================================
fig, ax = plt.subplot_mosaic(
    """
    AAABBB
    CCCDDD
    """,
    figsize=(12, 10),
)

# ---------------------------------- A --------------------------------------
# Grouped bars: passing a LIST of arrays to hist() draws the three conditions
# side by side within each bin rather than stacked or overlaid.
# Fewer bins than an overlay would use, since each bin now holds three bars.
bins = np.linspace(self_kd["self_log2fc"].min(),
                   self_kd["self_log2fc"].max(), 30)

data   = [self_kd.loc[self_kd["condition"] == c, "self_log2fc"].values
          for c in cond_order]
labels = [f"{c} (median {np.median(d):.2f})" for c, d in zip(cond_order, data)]
colors = [pal.get(c, "#888") for c in cond_order]

ax["A"].hist(data, bins=bins, label=labels, color=colors,
             edgecolor="none", rwidth=0.85)

ax["A"].axvline(0, c="k", lw=1)
ax["A"].set_xlabel("self log2FC — target transcript, KO vs condition-matched control")
ax["A"].set_ylabel("number of targets")
ax["A"].legend(fontsize=8)

n_down = (self_kd.groupby("perturbation")["self_log2fc"].median() < 0).sum()
n_tot  = self_kd["perturbation"].nunique()
ax["A"].text(0.02, 0.95,
             f"targets:        {n_tot}\n"
             f"median FC < 0:  {n_down} ({n_down/n_tot:.0%})\n"
             f"overall median: {self_kd['self_log2fc'].median():.2f}",
             transform=ax["A"].transAxes, ha="left", va="top",
             fontsize=8, family="monospace",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                       edgecolor="none", alpha=0.85))

for k in ["B", "C", "D"]:
    ax[k].set_axis_off()

titles = {"A": "self-knockdown by condition"}
for label, a in ax.items():
    a.set_title(f"{label}) {titles.get(label, '')}", loc="left",
                fontweight="bold", fontsize=12, pad=8)

fig.tight_layout()
savefig(fig, "02_perturbation_qc", cfg)

In [ ]:
# for B) we can look at ADT, for the surface proteins that were CRISPR-KO targets
# this is many fewer data points (ADTs relevant)

adt_genes = {g for genes in panels["adt"]["adt_to_rna"].values() for g in genes}
targets = set(rna.obs[PERT].unique()) - {CTRL}
sorted(adt_genes & targets)

In [ ]:
# ---------------------------------- B --------------------------------------
# RNA vs protein self-knockdown, for the 12 targets whose product is in the
# ADT panel. These are the only cases in the dataset where the same molecule
# is measured both ways AND we caused the change — so they calibrate how far
# RNA log2FC can be trusted as a proxy for protein for the other ~236 targets.
#
# Note: the HLA_A ADT feature is clone W6/32, a conformational pan-MHC-I
# epitope requiring B2M. HLA-A/B/C knockouts are therefore expected to move it
# only partially, since the antibody still detects the remaining two chains.

# ADT pseudobulk on the same (perturbation, condition) keys, CLR-normalised

ax["B"].clear()

import muon as mu
from adjustText import adjust_text

adt_n = adt.copy()
adt_n.layers["counts"] = adt_n.X.copy()
mu.prot.pp.clr(adt_n, axis=cfg["protein"]["clr_margin"])

adt_X = np.asarray(adt_n.X.todense()) if sp.issparse(adt_n.X) else np.asarray(adt_n.X)
adt_df = pd.DataFrame(adt_X, index=adt_n.obs_names, columns=adt_n.var_names)
adt_df["_grp"] = groups                       # same keys as the RNA pseudobulk
adt_mean = adt_df.groupby("_grp", observed=True).mean()   # mean CLR, not sum

# invert adt_to_rna: gene symbol -> ADT feature
rna_to_adt = {}
for feat, genes in panels["adt"]["adt_to_rna"].items():
    for g in genes:
        rna_to_adt.setdefault(g, []).append(feat)

rows = []
for cond in cond_order:
    ctrl_key = f"{CTRL}|{cond}"
    for gene, feats in rna_to_adt.items():
        key = f"{gene}|{cond}"
        if key not in adt_mean.index or ctrl_key not in adt_mean.index:
            continue
        rna_row = self_kd[(self_kd["perturbation"] == gene) &
                          (self_kd["condition"] == cond)]
        if rna_row.empty:
            continue
        for feat in feats:
            rows.append({
                "gene": gene,
                "adt_feature": feat,
                "condition": cond,
                "rna_log2fc": rna_row["self_log2fc"].iloc[0],
                # CLR is already log-scale, so a difference IS a log ratio
                "adt_delta": adt_mean.loc[key, feat] - adt_mean.loc[ctrl_key, feat],
                "n_cells": int(rna_row["n_cells"].iloc[0]),
            })

rna_vs_adt = pd.DataFrame(rows)
print(f"{rna_vs_adt['gene'].nunique()} genes x {len(cond_order)} conditions "
      f"= {len(rna_vs_adt)} paired measurements")

for cond in cond_order:
    d = rna_vs_adt[rna_vs_adt["condition"] == cond]
    ax["B"].scatter(d["rna_log2fc"], d["adt_delta"], s=45, alpha=0.85,
                    color=pal.get(cond, "#888"), label=cond,
                    edgecolor="white", linewidth=0.5)

ax["B"].axhline(0, c="k", lw=0.8, ls=":")
ax["B"].axvline(0, c="k", lw=0.8, ls=":")
lim = max(abs(np.r_[ax["B"].get_xlim(), ax["B"].get_ylim()]))
ax["B"].plot([-lim, lim], [-lim, lim], ls="--", c="grey", lw=1, label="concordance")
ax["B"].set_xlim(-lim, lim); ax["B"].set_ylim(-lim, lim)
ax["B"].set_xlabel("RNA self log2FC")
ax["B"].set_ylabel("ADT self Δ (CLR)")
ax["B"].legend(fontsize=8, loc="upper right")

texts = []
for gene, d in rna_vs_adt.groupby("gene"):
    cx, cy = d["rna_log2fc"].mean(), d["adt_delta"].mean()
    texts.append(ax["B"].text(cx, cy, d["adt_feature"].iloc[0], fontsize=7))

adjust_text(texts, ax=ax["B"],
            expand=(2.4, 2.8),          # was (1.6, 1.8) — bigger label bounding box
            force_text=(1.5, 2.0),      # was (0.6, 0.9) — labels shove each other harder
            force_static=(0.8, 1.2),    # was (0.3, 0.5) — pushed further off the points
            force_explode=(0.4, 0.6),   # initial scatter before optimising
            max_move=80,                # allow labels to travel further from origin
            iter_lim=200,               # more optimisation passes
            ensure_inside_axes=True)

# adjust_text has now moved each label; read its final position and connect it
# to all three of that gene's points.
for txt, (gene, d) in zip(texts, rna_vs_adt.groupby("gene")):
    lx, ly = txt.get_position()
    for _, r in d.iterrows():
        ax["B"].plot([lx, r["rna_log2fc"]], [ly, r["adt_delta"]],
                     ls="--", lw=0.4, color="grey", zorder=0)

r = rna_vs_adt[["rna_log2fc", "adt_delta"]].corr().iloc[0, 1]
ax["B"].text(0.97, 0.03, f"Pearson r = {r:.2f}\nn = {len(rna_vs_adt)}",
             transform=ax["B"].transAxes, ha="right", va="bottom",
             fontsize=8, family="monospace",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                       edgecolor="none", alpha=0.85))

fig.tight_layout()
savefig(fig, "02_perturbation_qc", cfg)
fig

In [ ]:
# ---------------------------------- C --------------------------------------
# Guide concordance. sgRNA is the replicate unit for the DE model, so this
# checks the assumption that unit rests on: do independent guides against the
# same gene produce the same knockdown?
#
# Same cheap pseudobulk as panel A, keyed on guide|condition instead of
# gene|condition. Guides below a minimum cell count are excluded — a guide
# with 8 cells gives a noisy estimate that reads as discordance when it is
# only sampling noise.
ax["C"].clear()

MIN_CELLS_PER_GUIDE = 10

guide_to_gene = (rna.obs[[GUIDE, PERT]].astype(str).drop_duplicates()
                 .set_index(GUIDE)[PERT].to_dict())

# pseudobulk keyed on guide|condition
g_groups = (rna.obs[GUIDE].astype(str) + "|" + rna.obs[COND].astype(str)).values
g_uniq   = pd.unique(g_groups)
g_row    = {g: i for i, g in enumerate(g_uniq)}

Mg = sp.csr_matrix(
    (np.ones(len(g_groups)), ([g_row[g] for g in g_groups], np.arange(len(g_groups)))),
    shape=(len(g_uniq), rna.n_obs),
)
pb_g = Mg @ counts
pb_g = np.asarray(pb_g.todense()) if sp.issparse(pb_g) else np.asarray(pb_g)
cpm_g = pd.DataFrame(pb_g / pb_g.sum(axis=1, keepdims=True) * 1e6,
                     index=g_uniq, columns=rna.var_names)
g_n = pd.Series(g_groups).value_counts()

rows = []
for key in g_uniq:
    guide, cond = key.rsplit("|", 1)
    gene = guide_to_gene.get(guide)
    if gene in (None, CTRL) or gene not in gene_ix:
        continue                                  # control guides / target not measured
    if g_n[key] < MIN_CELLS_PER_GUIDE:
        continue
    ctrl_key = f"{CTRL}|{cond}"
    if ctrl_key not in cpm.index:
        continue
    rows.append({
        "gene": gene, "guide": guide, "condition": cond,
        "n_cells": int(g_n[key]),
        "self_log2fc": np.log2((cpm_g.loc[key, gene] + 1) /
                               (cpm.loc[ctrl_key, gene] + 1)),
    })

guide_kd = pd.DataFrame(rows)
print(f"{len(guide_kd)} guide x condition estimates "
      f"(>= {MIN_CELLS_PER_GUIDE} cells), "
      f"{guide_kd['gene'].nunique()} genes, "
      f"{guide_kd['guide'].nunique()} guides")

# collapse conditions — editing preceded treatment, so pool for this view
gk = guide_kd.groupby(["gene", "guide"])["self_log2fc"].mean().reset_index()
gene_med = gk.groupby("gene")["self_log2fc"].median().sort_values()
gk["x"] = gk["gene"].map({g: i for i, g in enumerate(gene_med.index)})

ax["C"].scatter(gk["x"], gk["self_log2fc"], s=7, alpha=0.55,
                color="#1F6FB2", edgecolor="none", label="individual guide")
ax["C"].plot(range(len(gene_med)), gene_med.values, lw=1.2, color="#0B3D6B",
             label="gene median")
ax["C"].axhline(0, c="k", lw=0.8, ls=":")
ax["C"].set_xlabel(f"target gene ({len(gene_med)}), sorted by median self log2FC")
ax["C"].set_ylabel("self log2FC (per guide)")
ax["C"].legend(fontsize=8, loc="lower right")

spread = gk.groupby("gene")["self_log2fc"].agg(lambda v: v.max() - v.min())
ax["C"].text(0.02, 0.05,
             f"genes:          {len(gene_med)}\n"
             f"median spread:  {spread.median():.2f}\n"
             f"spread > 1.0:   {(spread > 1).sum()}",
             transform=ax["C"].transAxes, ha="left", va="bottom",
             fontsize=8, family="monospace",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                       edgecolor="none", alpha=0.85))


# ---------------------------------- D --------------------------------------
# Guide eligibility. The DE model treats guides as replicates, so a
# (gene, condition) contrast is only estimable if at least two independent
# guides cleared the cell-count minimum in THAT condition. This is a design
# requirement rather than a post-hoc filter: a contrast with one usable guide
# has no within-group variance to estimate.
#
# Computed per condition, not per gene — a gene can be well covered in control
# and thin in co-culture, and that asymmetry is exactly what matters.
ax["D"].clear()

MIN_GUIDES = 2

elig = (guide_kd.groupby(["gene", "condition"])["guide"].nunique()
        .unstack(fill_value=0).reindex(columns=cond_order, fill_value=0))

counts_by_n = pd.DataFrame({
    cond: elig[cond].value_counts().reindex(range(0, 4), fill_value=0)
    for cond in cond_order
})

x = np.arange(len(counts_by_n.index))
w = 0.26
for j, cond in enumerate(cond_order):
    ax["D"].bar(x + (j - 1) * w, counts_by_n[cond], width=w,
                label=cond, color=pal.get(cond, "#888"))

ax["D"].axvline(MIN_GUIDES - 0.5, ls="--", c="crimson", lw=1.2)
ax["D"].set_xticks(x)
ax["D"].set_xticklabels(counts_by_n.index)
ax["D"].set_xlabel(f"usable guides of 3 (>= {MIN_CELLS_PER_GUIDE} cells) per gene, within condition")
ax["D"].set_ylabel("number of genes")
ax["D"].legend(fontsize=8)

testable = (elig >= MIN_GUIDES)
all_three = testable.all(axis=1).sum()
ax["D"].text(0.97, 0.95,
             f"contrasts testable:  {testable.values.sum()} / {testable.size}\n"
             f"genes testable in\n"
             f"  all 3 conditions:  {all_three} / {len(elig)}",
             transform=ax["D"].transAxes, ha="right", va="top",
             fontsize=8, family="monospace",
             bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                       edgecolor="none", alpha=0.85))

# save the eligibility table — nb03 needs to know which contrasts are real
elig.to_csv(P.tables / "02_guide_eligibility.csv")

titles = {
    "A": "self-knockdown by condition",
    "B": "RNA vs protein self-knockdown",
    "C": "guide concordance within target",
    "D": "contrasts with enough independent guides",
}
for label, a in ax.items():
    a.set_title(f"{label}) {titles.get(label, '')}", loc="left",
                fontweight="bold", fontsize=12, pad=8)

fig.tight_layout()
savefig(fig, "02_perturbation_qc", cfg)
fig

In [ ]:
gk_n = guide_kd.groupby("gene")["n_cells"].min()
spread_vs_n = pd.DataFrame({"spread": spread, "min_cells": gk_n}).dropna()
print(spread_vs_n.corr(method="spearman"))

In [ ]:
rna.obs[GUIDE].unique()[:5]

In [ ]:
rna.obs[rna.obs[PERT] == "IFNGR1"][GUIDE].value_counts()

In [ ]:
self_kd.to_csv(P.tables / "02_self_knockdown_gene.csv", index=False)
guide_kd.to_csv(P.tables / "02_self_knockdown_guide.csv", index=False)
rna_vs_adt.to_csv(P.tables / "02_rna_vs_adt_selfkd.csv", index=False)
# elig already written

# the two interpretive flags, for downstream annotation
flags = pd.DataFrame({"guide_spread": spread})
flags["high_spread"] = flags["guide_spread"] > 1.0
flags.to_csv(P.tables / "02_gene_flags.csv")
print(f"flagged high-spread genes: {flags['high_spread'].sum()}")